[![Logo ><](../../Auxiliary_Files/Graphics/Databases/nea_logo.png)](https://exoplanetarchive.ipac.caltech.edu/)
## Table of Contents
1. [Overview](#overview)
2. [Accessing Data](#accessing-data)
    - [Web Interface](#web-interface)
        - [Data Columns](#data-columns)
        - [Uncertainties](#uncertainties)
        - [Filtering Methods](#filtering-methods)
        - [Downloading Tables](#downloading-tables)
        - [Searchable Tables](#searchable-tables)
    - [NASA Exoplanet Archive API via Astroquery](#nasa-exoplanet-archive-api-via-astroquery)
        - [Single Object Query](#single-object-query)
        - [Query by Region](#query-by-region)
        - [Query by Criteria](#query-by-criteria)
        - [All Available Tables](#all-available-tables)
    - [Table Access Protocol (TAP)](#table-access-protocol-tap)
        - [Building Queries](#building-queries)
    - [Table Documentation](#table-documentation)
        - [General](#general)
        - [Kepler](#kepler)
        - [Transit Surveys](#transit-surveys)
3. [Tools](#tools)
    - [Transit and Ephemeris Service](#transit-and-ephemeris-service)
    - [Confirmed Planet Plotting Tool](#confirmed-planet-plotting-tool)
    - [Predicted Observables for Exoplanets Service (POE)](#predicted-observables-for-exoplanets-service-poe)
4. [Exercises](#exercies)
    - [Exoplanet Distribution Across the Celestial Sphere](#1-exoplanet-distribution-across-the-celestial-sphere)
    - [Planning a JWST Observation](#2-planning-a-jwst-observation)
    - [Power Laws](#3-power-laws)


In [ ]:
from tdpy.plotting import configure_plot_style

typeplotback = 'white'
configure_plot_style(typeplotback)

# Overview
The [NASA Exoplanet Archive](https://exoplanetarchive.ipac.caltech.edu/) is an invaluable resource in exoplanet research. It hosts a variety of different tables with the most up-to-date astrophysical parameters of all known exoplanets and their hosts. Additionally, it hosts some mission data, as well as a few observer tools (e.g. transit and ephemeris calculator which aids in observational planning) that have great utility. This module briefly explains the main products of the site, how to use/navigate them, and best practices when using the observer tools as well as accessing data.

**Run the following code to initialize the interactive portions of this lesson:**

In [ ]:
from jupyterquiz import display_quiz
from astropy.coordinates import SkyCoord
import astropy.units as u
import warnings
import json
warnings.filterwarnings("ignore")
with open("../../Exercise_Solutions/Module_1/NEA/Checkpoints/questions.json", "r") as file:
    questions=json.load(file)

# Accessing Data

## Web Interface

Most catalogs in the Exoplanet Archive are organized as interactive tables. This overview covers the main features. The [complete user guide](https://exoplanetarchive.ipac.caltech.edu/docs/ICEexohelp.html) has additional detail.

### Data Columns

The [Planetary Systems Composite Data catalog](https://exoplanetarchive.ipac.caltech.edu/cgi-bin/TblView/nph-tblView?app=ExoTbls&config=PSCompPars) provides columns for the star, planet, full system, and related observations such as light curves. The standard view shows only a subset of available columns. Use the "Select Columns" tab to choose more.

![Table](../../Auxiliary_Files/Graphics/Databases/NEA_Table.png)

![Table](../../Auxiliary_Files/Graphics/Databases/NEA_Column.png)

Not every entry has a value in every column. Blank entries mean the corresponding measurement has not been reported to the archive.

### Uncertainties

Numerical columns often include uncertainties. Some measurements use symmetric uncertainties, written $x \\pm \\delta x$. Others have asymmetric upper and lower uncertainties, written $x^{+\\delta x_1}_{-\\delta x_2}$. These differences often reflect the methods used in the source paper. Some entries have no reported uncertainty. Read the cited literature to assess how well each value is constrained. Uncertainty values are also available in separate columns when you need to parse them independently.

### Filtering Methods

Use the filter box beneath a column header to narrow the table. Click the red question mark beside the header to see the syntax for that column.

![Table](../../Auxiliary_Files/Graphics/Databases/NEA_help.png)

Numeric columns support comparisons such as $<, >, \\leq, \\geq$ and inclusive ranges such as $[x,y]$. Integer columns use the same query syntax. Text columns support substring matching, substring exclusion, and the `%` wildcard. Use **null** to find empty entries and **not null** to find populated entries.

### Downloading Tables

Download the selected and filtered rows from the "Download Table" tab beside "Select Columns".

![Table](../../Auxiliary_Files/Graphics/Databases/NEA_COptions.png)

Choose a format such as CSV, VOTable, IPAC, or tab-separated values. You can also check which columns to include. By default, uncertainties are appended to the corresponding values in the same cell. The "Values Only" option removes uncertainties from those cells but keeps the separate uncertainty columns. Select the uncertainty columns if you want their values included in the download.

### Checkpoint 1

In [ ]:
##Run this code for checkpoint questions before continuing!
##To collapse the quiz once you are finished, click the 'o' key right under the play key to the left of the code block.
display_quiz([questions[0]] + [questions[1]] + [questions[2]])

## NASA Exoplanet Archive API via Astroquery

In some applications, systematically retrieving information from the NEA in Python is very useful. Here, we outline the basics of using the NEA and provide a couple of examples.

### Single Object Query
To start, we import the appropriate modules and query the exoplanet TRAPPIST-1, a red dwarf:

In [ ]:
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive as NEA

##Import the name, radius, and period of all planets around TRAPPIST-1
NEA.query_object('TRAPPIST-1', select='pl_name,pl_radj,pl_orbper')

This gives a table of relevant entries and parameters for planets around TRAPPIST-1. You can define the columns you wish to see using the `select` function, though filtering specific parameters is not supported for this method.
### Query by Region
For survey missions, knowing the available target list in a region can be useful. Astroquery lets you do this using `query_region`. To make the query easier, we will also import the `SkyCoord` module from `astropy`, allowing us to convert astrometry values seamlessly:

In [ ]:
## Using RA and DEC in degrees. This looks in a 0.5 degree circle centered at 100 degrees RA and 25 degrees declination.
NEA.query_region(
    table="pscomppars", coordinates=SkyCoord(ra=100 * u.deg, dec=25 * u.deg),
    radius=0.5 * u.deg) 

### Query by Criteria
The above methods will give you all available data corresponding to the target from all archive tables. Sometimes more specificity is required for particular parameters, such as the table it is pulled from, or all exoplanets meeting specific criteria. This can be done using the `query_criteria` method. This can be coupled with the `where`, `select`, and `order` parameters to get highly specific results. All available parameters are described [here](https://exoplanetarchive.ipac.caltech.edu/docs/program_interfaces.html) under the **Keywords for Specific Queries** section.

In [ ]:
#Here, select means a particular column, and gives a cumulative count.
#'Where' is querying all planets found by the K2 mission
NEA.query_criteria(table="pscomppars", select="count(*)",
                                    where="disc_facility like '%K2%'")  

In [ ]:
#Here, 'select' queries the planet radius column.
#'Where' is querying all planets found by microlensing
NEA.query_criteria(
    table="pscomppars", select="pl_radj", where="discoverymethod like 'Microlensing'")  

### All Available Tables
To find all accessible tables using the API, use the `TAP_TABLES` method:

In [ ]:
NEA.TAP_TABLES

### Checkpoint 2

In [ ]:
##Run this code for checkpoint questions before continuing!
##To collapse the quiz once you are finished, click the 'o' key right under the play key to the left of the code block.
display_quiz([questions[3]] + [questions[4]])

In [ ]:
##Codespace for Checkpoint 2

## Table Documentation

The Exoplanet Archive documents each table's columns and query fields. The links below identify the Table Access Protocol (TAP) name when one is available.

#### General
- [Planetary Systems and Planetary Systems Composite Parameters Table (ps, pscomppars)](https://exoplanetarchive.ipac.caltech.edu/docs/API_PS_columns.html#columns)
- [Microlensing Planets Table (ML)](https://exoplanetarchive.ipac.caltech.edu/docs/API_ML.html)
- [Direct Imaging Table](https://exoplanetarchive.ipac.caltech.edu/docs/API_directimaging.html)
- [Atmospheric Spectroscopy Table](https://exoplanetarchive.ipac.caltech.edu/cgi-bin/atmospheres/nph-firefly?atmospheres)
- [Transiting Planets Table](https://exoplanetarchive.ipac.caltech.edu/docs/API_transit_detection.html)

#### Kepler
- [Kepler Objects of Interest Tables (cumulative)](https://exoplanetarchive.ipac.caltech.edu/docs/API_kepcandidate_columns.html)
- [Kepler Objects of Interest Positional Probabilities Table](https://exoplanetarchive.ipac.caltech.edu/docs/API_koiapp_columns.html)
- [Kepler Astrophysical False Positive Probabilities Table](https://exoplanetarchive.ipac.caltech.edu/docs/API_fpp_columns.html)
- [Kepler Certified False Positive Table](https://exoplanetarchive.ipac.caltech.edu/docs/API_fpwg_columns.html)
- [Kepler Threshold-Crossing Event Table](https://exoplanetarchive.ipac.caltech.edu/docs/API_tce_columns.html)
- [Kepler Stellar Table](https://exoplanetarchive.ipac.caltech.edu/docs/API_keplerstellar_columns.html)

#### Transit Surveys
- [TESS Objects of Interest Table (TOI)](https://exoplanetarchive.ipac.caltech.edu/docs/API_TOI_columns.html)
- [K2 Planets and Candidates Table (k2pandc)](https://exoplanetarchive.ipac.caltech.edu/docs/API_k2pandc_columns.html)
- [K2 Targets Table (k2targets)](https://exoplanetarchive.ipac.caltech.edu/docs/)
- [KELT Table (kelttimeseries)](https://exoplanetarchive.ipac.caltech.edu/applications/ETSSView/docs/ETSS_KELTP_columns.html)
- [CoRoT Search Table](https://exoplanetarchive.ipac.caltech.edu/docs/datasethelp/ETSS_CoRoT.html)
- [XO Table](https://exoplanetarchive.ipac.caltech.edu/applications/ETSSView/docs/ETSS_XO_columns.html)
- [Cluster Table](https://exoplanetarchive.ipac.caltech.edu/applications/ETSSView/docs/ETSS_Cluster_columns.html)
- [TrES Table](https://exoplanetarchive.ipac.caltech.edu/applications/ETSSView/docs/ETSS_TrES_columns.html)
- [Mission Star and HWO ExEP Tables (di_stars_exep)](https://exoplanetarchive.ipac.caltech.edu/docs/API_mission_stars.html)

# [Tools](https://exoplanetarchive.ipac.caltech.edu/docs/tools.html)
The archive hosts some useful tools which can help researchers visualize data, as well as perform cursory analysis. This section aims to briefly cover their basic functionality.

## [Transit and Ephemeris Service](https://exoplanetarchive.ipac.caltech.edu/cgi-bin/TransitView/nph-visibletbls?dataset=transits)
Several research areas depend on accurate observations of transiting exoplanets (such as atmospheric characterization). These observations need to planned precisely so that they occur when the planet is transiting. To aid in this process, the NASA Exoplanet Archive hosts a transit and ephemeris service that uses known planetary periods, telescope locations, and transit epochs to develop transit windows to be used in observation planning. This subsection outlines a simple use case. For the user guide, click the 'User Guide' tab.

### Target Selection

The tool allows for three main modes of target selection:
- **Multiple Targets**: Used in most cases. Can draw ephemerides from existing tables, such as the Composite Planetary Systems table, the TESS Objects of Interest (TOI) table, the Kepler table, etc, or from a custom list of target names.
- **Single Target**: Find the ephemerides associated with a single target (i.e. 'TOI 210', 'Kepler 1020') or a single host (i.e. 'Proxima Centauri'). This will return multiple ephemerids if the host has multiple planets.
- **User Defined**: If the target of interest is not yet available in the archive, or the archive has not been updated with new findings, you can pass your own set of planetary parameter. NOTE: If you want to use a specific observatory, you will need to pass coordinates of some sort!

### Observer Location

The observer location depends on which instrument you plan to use. This is explanatory, and also allows putting in your own coordinates if you plan to do amateur observations.

### Observing Window and Phases

Lastly, if you have a specific time window to make the observation, you can input the observing window of interest to see a list of events that will occur. Additionally, you can query when in the transit you wish to observe, i.e. 0 phase (primary transit), 0.5 phase (secondary transit), or anywhere in between. Note that the time window is in Universal Time, so be sure to convert accordingly,

### Example Query

Say I am interested in determining the effective temperature of WASP-18b, which requires observing the **secondary** transit (also known as an *occultation*), where the brightness of the star-planet system dips due to the planet moving behind the star. Say I want to do this using the James Webb Space Telescope, and I have time for the month of December 2024. Here is the query:
- **Single Target**:

![Table](../../Auxiliary_Files/Graphics/Databases/NEA_Ephemerids.png)

Note the different colors of the entries, representing the uncertainty confidence for the corresponding ephemerids. We want the most precise option, so we double-click on the *ExoFOP-TESS TOI* entry, which automatically fills in the 'User Defined' parameters.
- **Observer Location: JWST, Observing Window: Dec. 2024, Phases: Secondary**:

![Table](../../Auxiliary_Files/Graphics/Databases/NEA_Ephermerid2.png)

This shows the rest of the query, including the observatory, time window, and desired phases.

### Output
Here is the output:

![Table](../../Auxiliary_Files/Graphics/Databases/NEA_Ephermerid_Output.png)

The columns selected are some of the relevant observing features, but not all. All available column definitions can be found [here](https://exoplanetarchive.ipac.caltech.edu/docs/transit/transit_parameters.html). We note that WASP-18b is viewable nearly every day, consistent with its orbital period. Other important aspects include the 'Transit Duration', 'Event Midpoint' (in this case of the secondary transit), and the 'Propagated Midpoint Uncertainty.' When reviewing your data, these are important parameters to make sure your data is consistent with the known exoplanet parameters.

# [Tools](https://exoplanetarchive.ipac.caltech.edu/docs/tools.html)
The archive hosts some useful tools which can help researchers visualize data, as well as perform cursory analysis. This section aims to briefly cover their basic functionality.

## [Transit and Ephemeris Service](https://exoplanetarchive.ipac.caltech.edu/cgi-bin/TransitView/nph-visibletbls?dataset=transits)
Several research areas depend on accurate observations of transiting exoplanets (such as atmospheric characterization). These observations need to planned precisely so that they occur when the planet is transiting. To aid in this process, the NASA Exoplanet Archive hosts a transit and ephemeris service that uses known planetary periods, telescope locations, and transit epochs to develop transit windows to be used in observation planning. This subsection outlines a simple use case. For the user guide, click the 'User Guide' tab.

### Target Selection

The tool allows for three main modes of target selection:
- **Multiple Targets**: Used in most cases. Can draw ephemerides from existing tables, such as the Composite Planetary Systems table, the TESS Objects of Interest (TOI) table, the Kepler table, etc, or from a custom list of target names.
- **Single Target**: Find the ephemerides associated with a single target (i.e. 'TOI 210', 'Kepler 1020') or a single host (i.e. 'Proxima Centauri'). This will return multiple ephemerids if the host has multiple planets.
- **User Defined**: If the target of interest is not yet available in the archive, or the archive has not been updated with new findings, you can pass your own set of planetary parameter. NOTE: If you want to use a specific observatory, you will need to pass coordinates of some sort!

### Observer Location

The observer location depends on which instrument you plan to use. This is explanatory, and also allows putting in your own coordinates if you plan to do amateur observations.

### Observing Window and Phases

Lastly, if you have a specific time window to make the observation, you can input the observing window of interest to see a list of events that will occur. Additionally, you can query when in the transit you wish to observe, i.e. 0 phase (primary transit), 0.5 phase (secondary transit), or anywhere in between. Note that the time window is in Universal Time, so be sure to convert accordingly,

### Example Query

Say I am interested in determining the effective temperature of WASP-18b, which requires observing the **secondary** transit (also known as an *occultation*), where the brightness of the star-planet system dips due to the planet moving behind the star. Say I want to do this using the James Webb Space Telescope, and I have time for the month of December 2024. Here is the query:
- **Single Target**:

![Table](../../Auxiliary_Files/Graphics/Databases/NEA_Ephemerids.png)

Note the different colors of the entries, representing the uncertainty confidence for the corresponding ephemerids. We want the most precise option, so we double-click on the *ExoFOP-TESS TOI* entry, which automatically fills in the 'User Defined' parameters.
- **Observer Location: JWST, Observing Window: Dec. 2024, Phases: Secondary**:

![Table](../../Auxiliary_Files/Graphics/Databases/NEA_Ephermerid2.png)

This shows the rest of the query, including the observatory, time window, and desired phases.

### Output
Here is the output:

![Table](../../Auxiliary_Files/Graphics/Databases/NEA_Ephermerid_Output.png)

The columns selected are some of the relevant observing features, but not all. All available column definitions can be found [here](https://exoplanetarchive.ipac.caltech.edu/docs/transit/transit_parameters.html). We note that WASP-18b is viewable nearly every day, consistent with its orbital period. Other important aspects include the 'Transit Duration', 'Event Midpoint' (in this case of the secondary transit), and the 'Propagated Midpoint Uncertainty.' When reviewing your data, these are important parameters to make sure your data is consistent with the known exoplanet parameters.

### [Confirmed Planet Plotting Tool](https://exoplanetarchive.ipac.caltech.edu/cgi-bin/IcePlotter/nph-icePlotInit?mode=demo&set=confirmed)
This tool allows you to plot any quantitative data from the interactive data tables, in either a scatter plot or a histogram. This can quickly generate basic visualizations between two parameters of interest, and has fairly robust editing options, such as log-log plots, binning, error bars, and labeling. Here is an example, plotting orbital period with orbital semi-major axis:

![Table](../../Auxiliary_Files/Graphics/Databases/NEA_Plot.png)

Kepler's law of period and orbital radius is quite apparent in this log-log plot. 

### [Predicted Observables for Exoplanets Service (POE)](https://exoplanetarchive.ipac.caltech.edu/cgi-bin/CalcQty/nph-calcqty)
This tool allows a user to determine theoretical observational signatures of a planet with certain constraints. The most commonly thought of constraint is **an Earth-like planet within its system's habitable zone**. Thus, given a target star or system, the tool will output the theoretical parameters the exoplanet must exhibit to meet those criteria. For example, what if you want to know the habitable zone range around Proxima Centauri? The input looks something like this:

![Table](../../Auxiliary_Files/Graphics/Databases/NEA_POE.png)

Giving an output data table of:

![Table](../../Auxiliary_Files/Graphics/Databases/NEA_POE_Out.png)

This shows the target with the input criteria (i.e., having the name Proxima Centauri) and the lower and upper bounds of the habitable zone, which are 0.03 AU and 0.0708 AU, respectively. Given that Proxima Centauri is a small, low-mass, and low-temperature star, it is reasonable that the habitable zone exists much closer than our Sun. This tool is very robust, so look at the in-depth [user guide](https://exoplanetarchive.ipac.caltech.edu/cgi-bin/CalcQty/nph-calcqty) for more information.

# Exercises

### 1. Heat Map of Microlensing Planets
Using `astroquery`, plot a heat map of the distribution of exoplanets discovered by the [microlensing](https://science.nasa.gov/mission/roman-space-telescope/microlensing/) method across the celestial sphere. That is, iterate over a grid of RA/DEC, count the number of micolensing planets in a square with length equal to the size of the grid spacing (e.g., if the grid spacing is 1 degree, search a cone of 1 degrees). What do you notice? What might explain this distribution? Use matplotlib's `imshow` method to create your plot.

[Sample Solution](../../Exercise_Solutions/Module_1/NEA/Exercises/Exercise_1.ipynb)

In [ ]:
# Codespace for Exercise 1
from matplotlib import pyplot as plt

### 2. Planning a JWST Observation
You are granted time on JWST's NIRCam in order to perform [transmission spectroscopy](https://www.esa.int/ESA_Multimedia/Images/2023/03/Transmission_spectroscopy) on a [hot Jupiter](https://en.wikipedia.org/wiki/Hot_Jupiter). Your observation block is from August 15th - August 20th, 2025. Plan an observation for JWST using the [ephemeris](https://exoplanetarchive.ipac.caltech.edu/cgi-bin/TransitView/nph-visibletbls?dataset=transits) tool by finding a planet that meets the following criteria:
- $T_{eq} > 1500 \; \mathrm{K}$
- $R_{p} > 1 \; R_{J}$
- $P < 10 \; \mathrm{Days}$

Additoinally, report the uncertainty in the transit time of your choice. Feel free to use the web interface or the Python API. Find the next feasible transit observation of your target.

[Sample Solution](../../Exercise_Solutions/Module_1/NEA/Exercises/Exercise_2.ipynb)

In [ ]:
# Codespace for Exercise 2
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive as NEA

### 3. Power Laws

Power-laws are ubiquitous in astrophysics, and exoplanet demographics are no exception, and they all follow the general expression:
$$y = x^{\alpha}$$

Using the NASA Exoplanet archive API, find the empirical $\alpha$ for the following parameter pairs, as well as a corresponding $\log-
\log$ plot:
- Period vs Semi-Major Axis, in Days and AU, respectively
- Mass vs Radius, in Earth masses/radii respectively, for planets below 100 Earth masses
- Mass vs Radius, for planets above 100 Earth masses
- Eccentricity vs Period

Comment on how well each trend fits. Be sure to include error bars! You can use `matplotlib`'s [`plt.errorbar`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.errorbar.html) to plot these easily.

HINT: You will want to use [`np.polynomial.polynomial.polyfit`](https://numpy.org/doc/stable/reference/generated/numpy.polynomial.polynomial.polyfit.html).

[Sample Solution](../../Exercise_Solutions/Module_1/NEA/Exercises/Exercise_3.ipynb)

In [ ]:
# Codespace for Exercise 3
import numpy as np
from matplotlib import pyplot as plt
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive as NEA